# BudgetSwitch W1 Day 6 — GRPO Smoke Test on Kaggle T4

**Goal:** Verify that Unsloth + TRL GRPO pipeline works end-to-end on Kaggle T4 with Qwen2.5-3B-Instruct in 4-bit, completing 5 steps without OOM or exceptions.

**This is NOT real training.** It uses a dummy reward function. The purpose is to confirm the pipeline is set up correctly before we invest 12 GPU-hours in W11.

**Expected runtime:** ~5-10 minutes on Kaggle T4.

**Success criteria:**
- ✅ All cells run without exception
- ✅ 5 GRPO steps complete
- ✅ Peak VRAM ≤ 14 GB
- ✅ Loss is computed (any finite value is acceptable; dummy reward = no real convergence)

**References:**
- Unsloth Qwen2.5 (3B) GRPO notebook: https://unsloth.ai/docs/get-started/reinforcement-learning-rl-guide
- Vi-S1K paper (Jan 2026): Vietnamese SLM scaling on Kaggle T4 — proof this resource profile works

## Setup: Install dependencies

Kaggle environments pre-install most things, but Unsloth + latest TRL need explicit install. Pin versions to avoid surprise breakages.

In [1]:
%%bash
# Cell 0: Fix libnvJitLink.so.13 — PHẢI chạy TRƯỚC cell install
# Tạo symlink vào system path để dynamic linker tìm được khi load bitsandbytes CUDA binary.
# Khác với os.environ/ctypes — cách này sửa ở cấp OS, hoạt động với dlopen().

set -e

echo "=== Tìm libnvJitLink ==="
LIB=$(find /usr/local/cuda* -name "libnvJitLink.so.13" 2>/dev/null | head -1)

if [ -z "$LIB" ]; then
    # Thử tìm bất kỳ version nào rồi dùng
    LIB=$(find /usr/local/cuda* -name "libnvJitLink.so*" 2>/dev/null | head -1)
fi

echo "Found: $LIB"

if [ -n "$LIB" ]; then
    ln -sf "$LIB" /usr/lib/x86_64-linux-gnu/libnvJitLink.so.13
    ldconfig
    echo "✅ Symlink tạo thành công + ldconfig cập nhật"
    ldconfig -p | grep nvJitLink
else
    echo "❌ Không tìm thấy libnvJitLink — liệt kê CUDA libs:"
    ls /usr/local/cuda*/lib64/*.so* 2>/dev/null | grep -i jit || echo "(không có)"
fi

=== Tìm libnvJitLink ===
Found: /usr/local/cuda-12.8/targets/x86_64-linux/lib/libnvJitLink.so
✅ Symlink tạo thành công + ldconfig cập nhật
	libnvJitLink.so.12 (libc6,x86-64) => /usr/local/cuda/targets/x86_64-linux/lib/libnvJitLink.so.12
	libnvJitLink.so.12 (libc6,x86-64) => /lib/x86_64-linux-gnu/libnvJitLink.so.12
	libnvJitLink.so (libc6,x86-64) => /usr/local/cuda/targets/x86_64-linux/lib/libnvJitLink.so


/sbin/ldconfig.real: /usr/local/lib/libtbbbind_2_5.so.3 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_adapter_level_zero.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbbind.so.3 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_adapter_level_zero_v2.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbbind_2_0.so.3 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_loader.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbb.so.12 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libumf.so.1 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtcm_debug.so.1 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_adapter_opencl.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbmalloc_proxy.so.2 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtcm.so.1 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/li

In [2]:
# Cell 1: Install - Fixed for Kaggle (CUDA 13 + unsloth-zoo compatibility)

# 1. Remove conflicting packages
!pip uninstall -y vllm torchvision torchaudio 2>/dev/null || true

# 2. Install Unsloth
!pip install --quiet "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git"

# 3. Chỉ upgrade peft + accelerate — KHÔNG upgrade transformers/trl/datasets
# (unsloth-zoo yêu cầu transformers<=5.5.0, trl<=0.24.0, datasets<4.4.0)
!pip install --quiet --upgrade peft accelerate

import os, torch
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Patch torch._inductor.cpp_builder TRƯỚC KHI import unsloth
# unsloth-zoo mới nhất dùng @torch_compile trong gemma3n.py, trigger import chain
# → torch 2.12.0+cu130 thiếu 'batch_convert_cubins_to_obj' → ImportError
# Fix: thêm function giả vào module (smoke test không cần compile CUDA kernels thật)
import torch._inductor.cpp_builder as _cpp_patch
if not hasattr(_cpp_patch, 'batch_convert_cubins_to_obj'):
    _cpp_patch.batch_convert_cubins_to_obj = lambda *a, **kw: []
    print("✅ Patched torch._inductor.cpp_builder (torch 2.12.0 compat)")

!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

import unsloth, trl, datasets, transformers
print(f"unsloth:      {unsloth.__version__}")
print(f"transformers: {transformers.__version__}")
print(f"trl:          {trl.__version__}")
print(f"datasets:     {datasets.__version__}")
try:
    import bitsandbytes
    print(f"bitsandbytes: {bitsandbytes.__version__}")
except Exception as e:
    print(f"bitsandbytes: load warning — {e}")

Found existing installation: torchvision 0.25.0+cu128
Uninstalling torchvision-0.25.0+cu128:
  Successfully uninstalled torchvision-0.25.0+cu128
Found existing installation: torchaudio 2.10.0+cu128
Uninstalling torchaudio-2.10.0+cu128:
  Successfully uninstalled torchaudio-2.10.0+cu128
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 868.6/868.6 kB 39.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 81.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 532.3/532.3 MB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.2/366.2 MB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.0/206.0 MB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━

W0521 16:02:33.243000 23 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0521 16:02:33.274000 23 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


🦥 Unsloth Zoo will now patch everything to make training faster!


<string>:1: FutureWarning: torch._dynamo.config.inline_inbuilt_nn_modules is deprecated and does not do anything, inline_inbuilt_nn_modules is always True. It will be removed in a future version of PyTorch.


unsloth:      2026.5.5
transformers: 5.5.0
trl:          0.24.0
datasets:     4.3.0
bitsandbytes: 0.49.2


## Step 1: Load Qwen2.5-3B with Unsloth

Use the pre-quantized 4-bit version to save download time. This is also what we'll use in W9-W11.

In [3]:
# Cell 2: Load model — 4-bit via Unsloth (hoạt động sau khi Cell 0 fix symlink)
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-3B-Instruct",
    max_seq_length=2048,
    load_in_4bit=True,       # ✅ dùng được sau khi ldconfig fix libnvJitLink
    fast_inference=False,    # vllm không available trên Kaggle CUDA 13
    dtype=None,
)

peak = torch.cuda.memory_allocated() / 1e9
print(f"✅ Model loaded (4-bit). VRAM: {peak:.2f} GB / 14.5 GB")

==((====))==  Unsloth 2026.5.5: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.12.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.7.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.36G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.42k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.36k [00:00<?, ?B/s]

... (2 outputs left out)

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/qwen2.5-3b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
✅ Model loaded (4-bit). VRAM: 2.40 GB / 14.5 GB


## Step 2: Add LoRA adapter (mode-routing simulation)

In real BudgetSwitch, this is the mode-routing LoRA (rank 8). Here we use same config to verify the setup.

**Note:** In W9-W11 real training, we'll have TWO LoRAs (tool-use frozen + mode-routing trainable). For smoke test, one LoRA is enough.

In [4]:
# Cell 3: Add LoRA
model = FastLanguageModel.get_peft_model(
    model,
    r=8,  # Mode-routing rank from proposal
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

# Count trainable params
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} ({100 * trainable / total:.4f}% of total)")

Not an error, but Unsloth cannot patch MLP layers with our manual autograd engine since either LoRA adapters
are not enabled or a bias term (like in Qwen) is used.
Unsloth 2026.5.5 patched 36 layers with 36 QKV layers, 36 O layers and 0 MLP layers.


Trainable params: 3,686,400 (0.2044% of total)


## Step 3: Build a tiny dummy dataset

10 toy prompts. Not Vietnamese — just verifying pipeline mechanics.

In [5]:
# Cell 4: Dummy dataset
from datasets import Dataset

DUMMY_PROMPTS = [
    "What is 2+2?",
    "Name the capital of France.",
    "Translate 'hello' to Spanish.",
    "What color is the sky?",
    "How many legs does a spider have?",
    "What is the largest planet?",
    "Who wrote Romeo and Juliet?",
    "What is H2O?",
    "Capital of Japan?",
    "5 times 5?",
]

def format_prompt(example):
    return {
        "prompt": [{"role": "user", "content": example["text"]}],
    }

dataset = Dataset.from_dict({"text": DUMMY_PROMPTS}).map(format_prompt)
print(f"Dataset: {len(dataset)} examples")
print("Sample:", dataset[0])

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

Dataset: 10 examples
Sample: {'text': 'What is 2+2?', 'prompt': [{'content': 'What is 2+2?', 'role': 'user'}]}


## Step 4: Dummy reward function

**IMPORTANT:** This is a DUMMY reward for pipeline testing only. The real BudgetSwitch reward (see `training/reward.py` in repo) uses outcome verifiers and follows CRITICAL_RULES.md.

Here: just return random rewards. We just need the pipeline to flow through.

In [6]:
# Cell 5: Dummy reward
import random

def dummy_reward(prompts, completions, **kwargs):
    """
    Dummy reward for pipeline smoke test.
    
    Real reward function will be in training/reward.py and will:
    1. Parse mode token from completion
    2. Execute step in chosen mode
    3. Run outcome verifier
    4. Compute R = r_outcome - β·cost + γ·r_final
    
    CRITICAL: Real reward MUST NOT reference oracle_mode.
    """
    return [random.uniform(0.0, 1.0) for _ in completions]

# Sanity test: call once
print("Reward test:", dummy_reward(["test"], ["completion"]))

Reward test: [0.6394267984578837]


## Step 5: Run GRPO for 5 steps

Small batch, small group size, just to verify mechanics.

In [7]:
# Cell 6: GRPO training (5 steps)
from trl import GRPOConfig, GRPOTrainer
import time

config = GRPOConfig(
    learning_rate=5e-6,
    num_generations=2,           # Small for smoke test (real: 4)
    max_completion_length=128,   # Small for smoke test
    max_steps=5,                 # JUST 5 STEPS
    per_device_train_batch_size=1,
    gradient_accumulation_steps=2,
    logging_steps=1,
    save_strategy="no",
    report_to="none",
    optim="adamw_torch",         # ← KHÔNG dùng adamw_8bit (cũng cần bitsandbytes CUDA)
    output_dir="/tmp/grpo_smoke_test",
    beta=0.04,
)

trainer = GRPOTrainer(
    model=model,
    args=config,
    reward_funcs=[dummy_reward],
    train_dataset=dataset,
    processing_class=tokenizer,
)

start_time = time.time()
trainer.train()
elapsed = time.time() - start_time

print(f"\n✅ GRPO smoke test completed in {elapsed:.1f} seconds.")
print(f"   Peak GPU memory during training: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 10 | Num Epochs = 1 | Total steps = 5
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 2 x 1) = 2
 "-____-"     Trainable parameters = 3,686,400 of 3,089,625,088 (0.12% trained)
Passing `generation_config` together with generation-related arguments=({'disable_compile', 'pad_token_id', 'cache_implementation'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will ta

Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / dummy_reward / mean,rewards / dummy_reward / std
1,0.000000,0.094770,0.053645,10.000000,10.000000,10.000000,0.000000,10.000000,10.000000,10.000000,0.000002,0.094770,0.053645
2,0.000000,0.259840,0.063532,8.000000,8.000000,8.000000,0.000000,8.000000,8.000000,8.000000,0.000000,0.259840,0.063532
3,0.000001,0.545476,0.425831,106.000000,84.000000,128.000000,0.500000,84.000000,84.000000,84.000000,0.000014,0.545476,0.425831
4,0.000001,0.777262,0.098336,128.000000,128.000000,128.000000,1.000000,0.000000,0.000000,0.000000,0.000017,0.777262,0.098336
5,0.000001,0.334396,0.102865,128.000000,128.000000,128.000000,1.000000,0.000000,0.000000,0.000000,0.000017,0.334396,0.102865


Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=1


✅ GRPO smoke test completed in 75.2 seconds.
   Peak GPU memory during training: 2.61 GB


## Success Criteria Verification

Run this final cell and capture screenshot for the W1 retrospective doc.

In [8]:
# Cell 7: Final verification
peak_vram_gb = torch.cuda.max_memory_allocated() / 1e9

print("=" * 50)
print("W1 Day 6 SMOKE TEST RESULTS")
print("=" * 50)
print(f"GRPO steps completed: 5 ✅")
print(f"Peak VRAM: {peak_vram_gb:.2f} GB  ({'✅ within budget' if peak_vram_gb <= 14 else '⚠️  OVER BUDGET'})")
print(f"Wall time: {elapsed:.1f}s ({elapsed/5:.1f}s per GRPO step)")
print("")
print("Extrapolation to full training (W11):")
print(f"  Real config will have group_size=4 (2x), max_completion=2048 (16x)")
print(f"  Estimated per-step time at full config: {elapsed/5 * 4 * 2:.1f}s")
print(f"  4500 steps × that = {4500 * elapsed/5 * 4 * 2 / 3600:.1f} hours total")
print("")
print("If estimate > 16 GPU-hours, we need to:")
print("  (a) reduce max_completion_length, or")
print("  (b) parallelize across 4 Kaggle accounts, or")
print("  (c) reduce training steps via curriculum.")

W1 Day 6 SMOKE TEST RESULTS
GRPO steps completed: 5 ✅
Peak VRAM: 2.61 GB  (✅ within budget)
Wall time: 75.2s (15.0s per GRPO step)

Extrapolation to full training (W11):
  Real config will have group_size=4 (2x), max_completion=2048 (16x)
  Estimated per-step time at full config: 120.3s
  4500 steps × that = 150.4 hours total

If estimate > 16 GPU-hours, we need to:
  (a) reduce max_completion_length, or
  (b) parallelize across 4 Kaggle accounts, or
  (c) reduce training steps via curriculum.


## What's Next

If this notebook runs end-to-end:
- ✅ Mark W1 Day 6 milestone done in retrospective
- ✅ Save this notebook to repo `notebooks/W1_grpo_smoke_test.ipynb`
- ✅ Document any version-specific issues in `docs/KAGGLE_NOTES.md`

If it fails:
- Try fallback 1: drop `fast_inference=True` (slower but more stable)
- Try fallback 2: use plain TRL GRPO without Unsloth wrapper
- Try fallback 3: switch to Colab Pro ($10/month, mentioned in Vi-S1K paper)

**Do not proceed to W2 if this smoke test fails.** The pipeline must work before we invest in data construction.